In [10]:
from pinecone import Pinecone,ServerlessSpec
import os
from dotenv import load_dotenv,find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings 

load_dotenv(find_dotenv(),override=True)
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [11]:
# List existing indexes
try:
    existing_indexes = pc.list_indexes()
    print("Existing indexes:", existing_indexes)
    index_names = [idx.name for idx in existing_indexes.indexes] if hasattr(existing_indexes, 'indexes') else []
except Exception as e:
    print(f"Error listing indexes: {e}")
    index_names = []

Existing indexes: [{
    "name": "testindex",
    "metric": "cosine",
    "host": "testindex-vudmgnn.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]


In [ ]:
# Only create index if it doesn't exist

index_name = "testindex"
if index_name not in index_names:
    print(f"Creating index '{index_name}'...")
    try:
        pc.create_index(
            name=index_name,
            dimension=384,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
        print(f"Index '{index_name}' created successfully!")
    except Exception as e:
        print(f"Error creating index: {e}")
        print(f"Error type: {type(e).__name__}")
else:
    print(f"Index '{index_name}' already exists!")

PineCone insert-1

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
index=pc.Index(index_name)
vectors_to_upsert = [
	{"id": "vec5", "values": embeddings.embed_query("my name is ajay"), "metadata": {"text": "india", "country": "india"}},
	# {"id": "vec2", "values": embeddings.embed_query("i am living in india"), "metadata": {"country": "india"}},
	# {"id": "vec3", "values": embeddings.embed_query("my village name is lahurpur"), "metadata": {"country": "india"}},
	# {"id": "vec4", "values": embeddings.embed_query("my city is allahabad"), "metadata": {"country": "india"}},
]

index.upsert(vectors=vectors_to_upsert, namespace="test-namespace")

PineCone insert 2

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_pinecone import PineconeVectorStore
load_dotenv(find_dotenv(),override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
index=pc.Index(index_name)
vectors_to_upsert = [
	{"id": "vec1", "values": embeddings.embed_query("my name is ram"), "metadata": {"country": "india","text":"my name is ram"}},
	{"id": "vec2", "values": embeddings.embed_query("i am living in india"), "metadata": {"country": "india","text":"i am living in india"}},
	{"id": "vec3", "values": embeddings.embed_query("example text for vec2"), "metadata": {"country": "india","text":"example text for vec2"}},
	{"id": "vec4", "values": embeddings.embed_query("example text for vec3"), "metadata": {"country": "india","text":"example text for vec3"}},
    {"id": "vec5", "values": embeddings.embed_query("my name is ajay"), "metadata": {"country": "japan","text":"my name is ajay"}},
	{"id": "vec6", "values": embeddings.embed_query("i am living in japan"), "metadata": {"country": "japan","text":"i am living in japan"}},
	{"id": "vec7", "values": embeddings.embed_query("example text for testing"), "metadata": {"country": "japan","text":"example text for testing"}},
	{"id": "vec8", "values": embeddings.embed_query("example text for testing "), "metadata": {"country": "japan","text":"example text for testing "}},
]

index.upsert(vectors=vectors_to_upsert, namespace="test-namespace")

Retrieve  Data

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_pinecone import PineconeVectorStore
load_dotenv(find_dotenv(),override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# vectorstore = PineconeVectorStore(index_name="testindex", embedding=embeddings, namespace="test-namespace")
resp=   index.query(
    vector=embeddings.embed_query("example text "),
    top_k=2,
    include_metadata=True,
    include_values=False,
    namespace="test-namespace",
    filter={"country": {"$eq": "india"}}
)
# len(docs)
resp

# Print the actual text content stored inside the metadata
for match in resp["matches"]:
   print("Matched Text:", match["metadata"])

PineconeVectorStore Retrieve Data

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_pinecone import PineconeVectorStore
load_dotenv(find_dotenv(),override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = PineconeVectorStore(index_name="testindex", embedding=embeddings, namespace="test-namespace")
docs=vectorstore.similarity_search( 
        "india",
        include_metadata=True,
        k=3
    )

page_contents: list[str] = [doc.page_content for doc in docs]

print(page_contents)

['i am living in india', 'i am living in japan', 'my name is ajay']


In [ ]:
ranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query="What is AAPL's outlook, considering both product launches and market conditions?",
    documents=[
        {"id": "vec2", "chunk_text": "Analysts suggest that AAPL'\''s upcoming Q4 product launch event might solidify its position in the premium smartphone market."},
        {"id": "vec3", "chunk_text": "AAPL'\''s strategic Q3 partnerships with semiconductor suppliers could mitigate component risks and stabilize iPhone production."},
        {"id": "vec1", "chunk_text": "AAPL reported a year-over-year revenue increase, expecting stronger Q3 demand for its flagship phones."},
    ],
    top_n=2,
    rank_fields=["chunk_text"],
    return_documents=True,
    parameters={
        "truncate": "END"
    }
)

ranked_results

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_pinecone import PineconeVectorStore
load_dotenv(find_dotenv(),override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = PineconeVectorStore(index_name="testindex", embedding=embeddings, namespace="test-namespace")
docs=vectorstore.similarity_search( 
        "india",
        include_metadata=True,
        k=3
    )

page_contents: list[str] = [doc.page_content for doc in docs]


ranked_results = pc.inference.rerank(
    model="pinecone-rerank-v0",
    query="india",
    documents=page_contents,
    top_n=2,
    rank_fields=["text"],
    return_documents=True,
    parameters={
        "truncate": "END"
    }
)

ranked_results

RerankResult(
  model='pinecone-rerank-v0',
  data=[{
    index=0,
    score=0.00055277866,
    document={
        text='i am living in india'
    }
  },{
    index=2,
    score=1.7778551e-05,
    document={
        text='my name is ajay'
    }
  }],
  usage={'rerank_units': 1}
)